# 非税收入分析

本Notebook用于分析各省份非税收入比例数据。

## 1. 环境配置与依赖导入

In [ ]:
# 导入标准库
import os
import sys
import datetime

# 导入数据处理库
import pandas as pd
import numpy as np

# 导入可视化库
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# 导入数据库库
import sqlalchemy
from sqlalchemy import create_engine

# 导入配置
from config import DATABASE_URL, INPUT_FILE, PROJECT_NAME

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print(f"项目: {PROJECT_NAME}")
print(f"当前时间: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. 数据加载与预览

In [ ]:
# 加载财政收入数据
# 注意：需要先将原始数据文件放到当前目录
data_path = INPUT_FILE

if os.path.exists(data_path):
    df = pd.read_excel(data_path)
    df.rename(columns={'指标名称': '日期'}, inplace=True)
    print(f"数据加载成功，共 {len(df)} 行")
    print(f"\n数据列: {df.columns.tolist()}")
    print(f"\n数据预览:")
    display(df.head(10))
else:
    print(f"数据文件 {data_path} 不存在")
    print("请将一般公共预算收入数据文件放到当前目录")
    df = None

## 3. 非税收入比例计算

In [ ]:
def calculate_non_tax_ratio(df):
    """
    计算各省份非税收入比例
    非税收入比例 = 1 - 税收收入/一般公共预算收入
    或 非税收入比例 = 非税收入/一般公共预算收入
    """
    non_tax_ratio_df = pd.DataFrame({'日期': pd.to_datetime(df['日期']).dt.strftime('%Y-%m-%d')})
    processed_provinces = set()
    
    for column in df.columns:
        if '税收' not in column and column != '日期':
            province = column.split(':')[0]
            
            if province in processed_provinces:
                continue
            
            general_revenue_col = column
            non_tax_cols = [col for col in df.columns if province in col and '非税' in col]
            
            if non_tax_cols:
                non_tax_revenue_col = non_tax_cols[0]
                non_tax_ratio = df[non_tax_revenue_col] / df[general_revenue_col]
            else:
                tax_cols = [col for col in df.columns if province in col and '税收' in col]
                tax_revenue_col = tax_cols[0] if tax_cols else None
                if tax_revenue_col:
                    non_tax_ratio = 1 - df[tax_revenue_col] / df[general_revenue_col]
                else:
                    continue
            
            non_tax_ratio_df[province] = non_tax_ratio
            processed_provinces.add(province)
    
    return non_tax_ratio_df

if df is not None:
    non_tax_ratio_df = calculate_non_tax_ratio(df)
    print("\n非税收入比例计算完成:")
    print(f"数据形状: {non_tax_ratio_df.shape}")
    print(f"省份列表: {non_tax_ratio_df.columns.tolist()}")
    display(non_tax_ratio_df.head())

## 4. 数据清洗与处理

In [ ]:
if 'non_tax_ratio_df' in dir() and non_tax_ratio_df is not None:
    # 转换数值类型
    for col in non_tax_ratio_df.columns:
        if col != '日期':
            non_tax_ratio_df[col] = pd.to_numeric(non_tax_ratio_df[col], errors='coerce')
    
    # 按日期排序
    non_tax_ratio_df.sort_values(by='日期', ascending=True, inplace=True)
    
    # 将负值替换为空值
    for col in non_tax_ratio_df.columns:
        if col != '日期':
            non_tax_ratio_df.loc[non_tax_ratio_df[col] < 0, col] = None
    
    # 前向填充
    non_tax_ratio_df.ffill(inplace=True)
    
    print("数据清洗完成")
    print(f"\n清洗后数据预览:")
    display(non_tax_ratio_df.head(10))

## 5. 数据分析与可视化

In [ ]:
# 非税收入比例时间序列图
if 'non_tax_ratio_df' in dir() and non_tax_ratio_df is not None:
    # 选择部分省份进行可视化
    provinces_to_plot = non_tax_ratio_df.columns[1:6].tolist()  # 前5个省份
    
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for province in provinces_to_plot:
        ax.plot(non_tax_ratio_df['日期'], non_tax_ratio_df[province], label=province, marker='o', markersize=2)
    
    ax.set_xlabel('日期')
    ax.set_ylabel('非税收入比例')
    ax.set_title('各省份非税收入比例变化趋势')
    ax.legend(loc='upper left')
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    os.makedirs('output', exist_ok=True)
    plt.savefig('output/non_tax_ratio_trend.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# 各省份非税收入比例热力图
if 'non_tax_ratio_df' in dir() and non_tax_ratio_df is not None:
    # 准备热力图数据
    heatmap_data = non_tax_ratio_df.set_index('日期').T
    heatmap_data = heatmap_data.iloc[:, -24:]  # 最近24个月
    
    fig, ax = plt.subplots(figsize=(16, 10))
    im = ax.imshow(heatmap_data.values, aspect='auto', cmap='RdYlGn_r')
    
    ax.set_yticks(range(len(heatmap_data.index)))
    ax.set_yticklabels(heatmap_data.index)
    ax.set_title('各省份非税收入比例热力图（最近24个月）')
    
    plt.colorbar(im, ax=ax, label='非税收入比例')
    plt.tight_layout()
    plt.savefig('output/non_tax_ratio_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. 数据库操作

In [ ]:
# 数据库连接
def get_db_connection():
    """获取数据库连接"""
    try:
        engine = create_engine(DATABASE_URL, poolclass=sqlalchemy.pool.NullPool)
        conn = engine.connect()
        print("数据库连接成功")
        return engine, conn
    except Exception as e:
        print(f"数据库连接失败: {e}")
        return None, None

engine, conn = get_db_connection()

In [ ]:
# 将非税收入比例数据写入数据库
if 'non_tax_ratio_df' in dir() and non_tax_ratio_df is not None and conn is not None:
    table_name = 'non_tax_ratio_df'
    try:
        non_tax_ratio_df.to_sql(table_name, con=conn, if_exists='replace', index=False)
        print(f"数据已成功写入表: {table_name}")
        print(f"写入记录数: {len(non_tax_ratio_df)}")
    except Exception as e:
        print(f"写入数据库失败: {e}")

## 7. 结果输出与总结

In [ ]:
# 生成输出报告
print("\n" + "="*60)
print(f"项目: {PROJECT_NAME} - 分析报告")
print("="*60)

if 'non_tax_ratio_df' in dir() and non_tax_ratio_df is not None:
    print(f"\n数据处理完成:")
    print(f"  - 数据行数: {len(non_tax_ratio_df)}")
    print(f"  - 省份数量: {len(non_tax_ratio_df.columns) - 1}")
    print(f"  - 省份列表: {non_tax_ratio_df.columns.tolist()[1:]}")
    
    # 保存处理后的数据
    output_file = 'output/non_tax_ratio_processed.xlsx'
    os.makedirs('output', exist_ok=True)
    non_tax_ratio_df.to_excel(output_file, index=False)
    print(f"\n处理后的数据已保存至: {output_file}")
else:
    print("\n无数据可处理")

print("\n" + "="*60)
print("分析完成！")
print("="*60)

In [ ]:
# 关闭数据库连接
if conn is not None:
    conn.close()
    print("数据库连接已关闭")